# Rede neural convolucional com transformada Wavelet no pipeline 


## 1. Dataset - MiniMIAS

### 1.1 Rodar localmente o programa

- Para isso é necessário baixar e extrair o dataset para a máquina local. 

**Link:** https://www.repository.cam.ac.uk/items/b6a97f0c-3b9b-40ad-8f18-3d121eef1459

- Baixe o dataset em Files;
- Após extrair em uma pasta os arquivos. Comando via terminal: _unzip MIAS-DB.zip_

In [5]:
import os

pasta = "dataset-mias/"

arquivos = os.listdir(pasta)
contador = 0

for arquivo in arquivos:
    if ".pgm" in arquivo:
        contador += 1

print(contador)

322


### 1.2 Classificando as imagnes

In [1]:
import os
import random
import shutil
from PyPDF2 import PdfReader

pasta_imagens = "dataset-mias/"

com_nodulo = []
sem_nodulo = []

# LER PDF
reader = PdfReader("dataset-mias/00README.pdf")
linhas = []

for pagina in reader.pages:
    texto = pagina.extract_text()
    linhas.extend(texto.split("\n"))

# PROCESSAR
for linha in linhas:
    if "mdb" in linha:
        partes = linha.split()
        nome = partes[0] + ".pgm"
        
        if "NORM" in linha:
            sem_nodulo.append(nome)
        else:
            com_nodulo.append(nome)

# DIVIDIR
def dividir(lista, pasta_train, pasta_test):
    random.shuffle(lista)
    corte = int(0.7 * len(lista))
    
    for nome in lista[:corte]:
        origem = os.path.join(pasta_imagens, nome)
        destino = os.path.join(pasta_train, nome)
        if os.path.exists(origem):
            shutil.copy(origem, destino)
    
    for nome in lista[corte:]:
        origem = os.path.join(pasta_imagens, nome)
        destino = os.path.join(pasta_test, nome)
        if os.path.exists(origem):
            shutil.copy(origem, destino)

# PASTAS
os.makedirs("dataset/train/com_nodulo", exist_ok=True)
os.makedirs("dataset/train/sem_nodulo", exist_ok=True)
os.makedirs("dataset/test/com_nodulo", exist_ok=True)
os.makedirs("dataset/test/sem_nodulo", exist_ok=True)

# EXECUTAR
dividir(com_nodulo, "dataset/train/com_nodulo", "dataset/test/com_nodulo")
dividir(sem_nodulo, "dataset/train/sem_nodulo", "dataset/test/sem_nodulo")

print("Finalizado!")

Finalizado!


### 1.3 Carregando dataset para CNN

In [2]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),  
    transforms.Resize((28, 28)), 
    transforms.ToTensor(),
])

train_dataset = datasets.ImageFolder(
    root="dataset/train",
    transform=transform
)

test_dataset = datasets.ImageFolder(
    root="dataset/test",
    transform=transform
)

# Criar DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

# Verificar Classes
print(train_dataset.classes)

['com_nodulo', 'sem_nodulo']


## 2. Criando uma camada Wavelet usando PyTorch

In [11]:
# Camada Wavelet 
import torch
import torch.nn as nn
import torch.nn.functional as F

class WaveletLayer(nn.Module):
    def __init__(self):
        super(WaveletLayer, self).__init__()

        # Filtro Haar LL
        self.kernel = torch.tensor([[0.5, 0.5],
                               [0.5, 0.5]], dtype=torch.float32)
    
    def forward(self, x):
        # Descobre quantos canais o dado de entrada (x) tem no momento
        in_channels = x.shape[1]
        
        # Cria o peso com o número correto de canais para este 'x'
        weight = self.kernel.repeat(in_channels, 1, 1, 1).to(x.device)
        
        # groups=in_channels faz a operação canal por canal 
        return F.conv2d(x, weight, stride=2, groups=in_channels)


## 3. Modelo com 3 convoluções de CNN integrando a camada Wavelet

<img src="img/waveletCNN-modelo3.png" width="700" title="Dica de ferramenta">

In [12]:
"""
MODELO COM 3 CONVOLUÇÕES E APLICANDO CAMADA WAVELET 
"""
class TripleWaveletCNN(nn.Module):
    def __init__(self):
        super(TripleWaveletCNN, self).__init__()
        
        # Camada Wavelet 
        self.wavelet = WaveletLayer() 
        
        # BLOCO 1
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        
        # BLOCO 2
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        
        # BLOCO 3
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        
        # Decisão Final
        # 28 -> 14 -> 7 -> 3. Logo: 128 filtros de 3x3
        self.fc1 = nn.Linear(128 * 3 * 3, 256)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(256, 2)

    def forward(self, x):
        # Passagem 1: 28x28 -> 14x14
        x = self.wavelet(x)
        x = torch.relu(self.bn1(self.conv1(x)))
        
        # Passagem 2: 14x14 -> 7x7
        x = self.wavelet(x) 
        
        x = torch.relu(self.bn2(self.conv2(x)))
        
        # Passagem 3: 7x7 -> 3x3
        x = self.wavelet(x)
        x = torch.relu(self.bn3(self.conv3(x)))
        
        # Vetorização e Classificação
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

### 3.1 Treinando o modelo

In [13]:
import torch
import torch.nn as nn
import torch.optim as optim

model = TripleWaveletCNN()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(50):
    model.train()
    
    for imagens, labels in train_loader:
        outputs = model(imagens)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch} - Loss: {loss.item()}")

Epoch 0 - Loss: 0.7079479694366455
Epoch 1 - Loss: 0.6340242028236389
Epoch 2 - Loss: 0.6388940215110779
Epoch 3 - Loss: 0.6644012331962585
Epoch 4 - Loss: 0.7493492364883423
Epoch 5 - Loss: 0.741015613079071
Epoch 6 - Loss: 0.6048278212547302
Epoch 7 - Loss: 0.6296437978744507
Epoch 8 - Loss: 0.607936441898346
Epoch 9 - Loss: 0.5861234068870544
Epoch 10 - Loss: 0.6232336163520813
Epoch 11 - Loss: 0.6176842451095581
Epoch 12 - Loss: 0.5143247246742249
Epoch 13 - Loss: 0.533323347568512
Epoch 14 - Loss: 0.7729796171188354
Epoch 15 - Loss: 0.502597451210022
Epoch 16 - Loss: 0.5343621969223022
Epoch 17 - Loss: 0.4672470986843109
Epoch 18 - Loss: 0.5274065732955933
Epoch 19 - Loss: 0.5218549370765686
Epoch 20 - Loss: 0.36429157853126526
Epoch 21 - Loss: 0.4151884913444519
Epoch 22 - Loss: 0.3524567782878876
Epoch 23 - Loss: 0.4086824059486389
Epoch 24 - Loss: 0.3456736207008362
Epoch 25 - Loss: 0.2565142512321472
Epoch 26 - Loss: 0.3664250075817108
Epoch 27 - Loss: 0.27910512685775757
Epoc

### 3.2 Testando o modelo

In [15]:
model.eval()
corretos = 0
total = 0

with torch.no_grad():
    for imagens, labels in test_loader:
        outputs = model(imagens)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        corretos += (predicted == labels).sum().item()

print(f"Acurácia: {100 * corretos / total:.2f}%")

Acurácia: 60.20%
